# Modelado Predictivo de Límite Elástico ($YS$ en MPa) en Aleaciones MPEAs
## Cuaderno de Experimentación, Optimización y Selección de Modelos

En este cuaderno se desarrolla el pipeline integral de modelado predictivo para el **Límite Elástico ($YS$ - *Yield Strength*)** en Aleaciones de Múltiples Elementos Principales (*Multi-Principal Element Alloys*, MPEAs).

### Flujo Metodológico:
1. **Configuración del Entorno y Carga de Datos**: Detección y lectura local de la matriz numérica featurizada (`dataset_final_ml.csv`).
2. **Partición de Datos**: División reproducible 80% entrenamiento y 20% prueba con semilla fija (`random_state=42`) para evitar fuga de información (*data leakage*).
3. **Benchmarking de Modelos**: Comparativa entre *Random Forest Regressor* (Bagging) y *XGBoost Regressor* (Gradient Boosting).
4. **Análisis de Explicabilidad e Importancia de Variables**: Consenso ponderado de importancia física (promedio RF + XGBoost) y visualización del Top 15.
5. **Reducción Dimensional (Barrido Grueso)**: Análisis de sensibilidad de 10 a 100 descriptores para ubicar el valle de error global.
6. **Optimización Fina (Micro-Barrido)**: Zoom de alta precisión (50 a 71 variables) para localizar el *Punto Dulce* exacto.
7. **Entrenamiento Final y Serialización**: Ajuste del modelo de producción con 51 variables y exportación de archivos binarios (`.pkl`) y metadata estandarizada (`.json`).

## Paso 1: Configuración de Librerías y Compatibilidad de Entorno

Importamos las librerías necesarias para el procesamiento matricial, modelado de ensamble, evaluación métrica y visualización. Se implementa una configuración que permite la ejecución tanto en entornos locales (JupyterLab, VS Code, terminal) como en Google Colab sin interrupciones por dependencias faltantes.

In [ ]:
import os
import json
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

# Compatibilidad con Google Colab (opcional si se ejecuta en la nube)
try:
    from google.colab import drive
    # drive.mount('/content/drive')
except ImportError:
    pass

## Paso 2: Carga de la Matriz Numérica y Partición de Datos (Train/Test Split)

Se carga la matriz `dataset_final_ml.csv` (1,008 muestras x 306 columnas), la cual contiene los descriptores composicionales extraídos con `matminer` y las variables categóricas codificadas mediante One-Hot Encoding binario (`dtype=int`).

Se separa la variable dependiente `Limite_Elastico_YS_MPa` de las 305 variables predictoras ($X$), y se realiza la partición estratificada por muestreo aleatorio en:
- **Conjunto de Entrenamiento (80%)**: Utilizado para el ajuste de pesos y árboles.
- **Conjunto de Prueba (20%)**: Datos reservados e inéditos para evaluar la capacidad real de generalización.

In [ ]:
# 1. Cargar la matriz matemática desde el entorno local
posibles_rutas_csv = [
    Path("dataset_final_ml.csv"),
    Path("Limite_elastico") / "dataset_final_ml.csv",
    Path("..") / "Limite_elastico" / "dataset_final_ml.csv"
]
ruta_csv = next((p for p in posibles_rutas_csv if p.is_file()), None)

if ruta_csv is None:
    raise FileNotFoundError(
        "No se encontró 'dataset_final_ml.csv'. Verifique que el archivo exista en la carpeta del notebook o en la raíz del proyecto."
    )

print(f"Cargando dataset desde: {ruta_csv.resolve()}")
df = pd.read_csv(ruta_csv)
print(f"Dimensiones de la matriz cargada: {df.shape[0]} filas x {df.shape[1]} columnas")

# 2. Separar características predictoras (X) y variable objetivo (y)
variable_objetivo = 'Limite_Elastico_YS_MPa'

X = df.drop(columns=[variable_objetivo])
y = df[variable_objetivo]

# 3. Partición de datos: 80% entrenamiento y 20% prueba (random_state=42 fijo para reproducibilidad)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Muestras para entrenamiento (80%): {X_train.shape[0]} | Muestras para prueba (20%): {X_test.shape[0]}")

## Paso 3: Definición e Instanciación de Modelos de Ensamble

Se configuran dos arquitecturas complementarias basadas en árboles de decisión:
1. **Random Forest Regressor**: Ensamble por agregación de arranque (*Bagging*), que construye múltiples árboles no correlacionados y promedia sus estimaciones, reduciendo la varianza.
2. **XGBoost Regressor**: Ensamble por potenciación de gradiente (*Gradient Boosting*), que construye árboles secuencialmente corrigiendo los residuos del árbol previo mediante funciones de pérdida regularizadas.

Ambos modelos se ejecutan en paralelo (`n_jobs=-1`) con 100 estimadores base.

In [ ]:
# 4. Instanciar los modelos de ensamble
# Se utiliza n_jobs=-1 para aprovechar todos los núcleos disponibles de CPU
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)

## Paso 4: Entrenamiento de los Modelos Base

Se ajustan los modelos sobre el 80% del dataset (`X_train`, `y_train`). Este proceso permite calibrar los umbrales de decisión y calcular las importancias intrínsecas de cada una de las 305 características.

In [ ]:
# 5. Entrenamiento de los modelos
print("Entrenando Random Forest Regressor...")
rf_model.fit(X_train, y_train)

print("Entrenando XGBoost Regressor...")
xgb_model.fit(X_train, y_train)
print("Entrenamiento completado exitosamente.")

## Paso 5: Predicciones y Evaluación Comparativa de Métricas

Se evalúa la precisión predictiva sobre el 20% de datos inéditos (`X_test`) utilizando:
- **$R^2$ (Coeficiente de Determinación)**: Evalúa la proporción de varianza explicada por el modelo respecto a la media.
- **$RMSE$ (Root Mean Squared Error)**: Cuantifica la desviación típica de los residuos en megapascales (MPa).

In [ ]:
# 6. Predicciones sobre el conjunto de prueba (20% no visto durante el entrenamiento)
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

# 7. Evaluación de métricas comparativas (R2 y RMSE)
rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
xgb_r2 = r2_score(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

print("
=== Resultados del Benchmarking Inicial ===")
print(f"Random Forest -> R2: {rf_r2:.4f} | RMSE: {rf_rmse:.2f} MPa")
print(f"XGBoost       -> R2: {xgb_r2:.4f} | RMSE: {xgb_rmse:.2f} MPa")

## Paso 6: Explicabilidad y Consenso de Importancia de Variables (Feature Importance)

Para evitar sesgos propios de la formulación de cada algoritmo (importancia por impureza de Gini en Random Forest vs. ganancia y frecuencia en XGBoost), se calcula la **Importancia Promedio de Consenso**.

A partir de este promedio, se extraen y grafican los **15 descriptores dominantes**, permitiendo corroborar el sentido físico del modelo:
- Propiedades atómicas de Magpie (volumen atómico medio, electronegatividad, radios covalentes).
- Contenido estequiométrico de aleantes clave ($Al$, $Ni$, $Mo$).
- Parámetros de solución sólida de Yang ($\delta$).
- Condiciones de ensayo y procesamiento (`Temperatura_Prueba_C`, `Tipo_Ensayo_C`, métodos pulvimetalúrgicos y microestructuras).

In [ ]:
# 1. Extraer importancias relativas de ambos modelos
df_importancias = pd.DataFrame({
    'Variable': X.columns,
    'Importancia_RF': rf_model.feature_importances_,
    'Importancia_XGB': xgb_model.feature_importances_
})

# 2. Calcular el consenso (promedio) para definir el verdadero Top 15
df_importancias['Importancia_Promedio'] = df_importancias[['Importancia_RF', 'Importancia_XGB']].mean(axis=1)
top_15_consenso = df_importancias.sort_values(by='Importancia_Promedio', ascending=False).head(15)

# 3. Reestructurar la tabla para visualización comparativa con Seaborn
df_grafico = top_15_consenso.melt(
    id_vars='Variable',
    value_vars=['Importancia_RF', 'Importancia_XGB'],
    var_name='Modelo',
    value_name='Nivel de Importancia'
)

# 4. Generar gráfico comparativo de barras
plt.figure(figsize=(12, 8))
sns.barplot(x='Nivel de Importancia', y='Variable', hue='Modelo', data=df_grafico, palette='Set2')
plt.title('Intersección de Variables Físicas: Random Forest vs XGBoost (Límite Elástico)', fontsize=14)
plt.xlabel('Peso / Nivel de Importancia en el Modelo', fontsize=12)
plt.ylabel('Descriptores Físico-Químicos', fontsize=12)
plt.tight_layout()
plt.show()

## Paso 7: Reducción Dimensional - Barrido Grueso (10 a 100 Variables)

Con 305 variables de entrada, el modelo corre riesgo de memorización y requiere un exceso de información en fase de producción.

Para optimizar el compromiso entre parsimonia (simplicidad) y capacidad predictiva, realizamos un **barrido descendente** evaluando el $RMSE$ al retener desde 10 hasta 100 variables en pasos de 10, guiados por el orden de importancia de XGBoost. Esto nos permite observar la existencia de una meseta de error mínimo alrededor de las 50 a 70 variables.

In [ ]:
# Extraer importancias ordenadas del modelo XGBoost base
importancias_xgb = xgb_model.feature_importances_
indices_ordenados = np.argsort(importancias_xgb)[::-1]

# Barrido grueso: de 10 a 100 variables, en pasos de 10
rango_variables = list(range(10, 110, 10))

errores_xgb = []
errores_rf = []

print("Calculando la curva de rendimiento y el valle de error (10 a 100 descriptores)...")

for n_vars in rango_variables:
    columnas_top = X.columns[indices_ordenados[:n_vars]]
    X_train_red = X_train[columnas_top]
    X_test_red = X_test[columnas_top]

    # Entrenar modelos rápidos
    xgb_tmp = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    rf_tmp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

    xgb_tmp.fit(X_train_red, y_train)
    rf_tmp.fit(X_train_red, y_train)

    # Registrar el RMSE en el conjunto de prueba
    errores_xgb.append(np.sqrt(mean_squared_error(y_test, xgb_tmp.predict(X_test_red))))
    errores_rf.append(np.sqrt(mean_squared_error(y_test, rf_tmp.predict(X_test_red))))

# Graficar curva de rendimiento
plt.figure(figsize=(10, 6))
plt.plot(rango_variables, errores_xgb, marker='o', linewidth=2, color='coral', label='XGBoost')
plt.plot(rango_variables, errores_rf, marker='s', linewidth=2, color='teal', label='Random Forest')

# Marcar punto dulce teórico exploratorio
plt.axvline(x=50, color='gray', linestyle='--', alpha=0.7, label='Región Óptima Estimada (~50 Vars)')

plt.title('Curva de Rendimiento: Cantidad de Variables vs RMSE (Límite Elástico)', fontsize=14)
plt.xlabel('Cantidad de Variables Retenidas (Top N)', fontsize=12)
plt.ylabel('Error RMSE (MPa) - Menor es mejor', fontsize=12)
plt.gca().invert_xaxis()  # Invertir eje X para lectura intuitiva de mayor a menor número de variables
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Paso 8: Micro-Barrido Fino - Localización Exacta del Punto Dulce (50 a 71 Variables)

Confirmado el valle de error en la zona de ~50 variables, ejecutamos un escáner granular de paso unitario (variable a variable) evaluando de 50 a 71 características.

Este procedimiento identifica el número exacto de descriptores que produce el mínimo absoluto de $RMSE$ en el conjunto de prueba.

In [ ]:
# Micro-barrido de alta resolución: iterar de 50 a 71, de 1 en 1
rango_fino = list(range(50, 72, 1))
errores_micro = []

print("Iniciando escáner de alta precisión con zoom al valle del error (50 a 71 variables)...")

for n_vars in rango_fino:
    columnas_top = X.columns[indices_ordenados[:n_vars]]
    X_train_red = X_train[columnas_top]
    X_test_red = X_test[columnas_top]

    xgb_tmp = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    xgb_tmp.fit(X_train_red, y_train)

    rmse_actual = np.sqrt(mean_squared_error(y_test, xgb_tmp.predict(X_test_red)))
    errores_micro.append(rmse_actual)

# Identificar el punto de error global mínimo
mejor_idx = np.argmin(errores_micro)
mejor_n = rango_fino[mejor_idx]
mejor_rmse = errores_micro[mejor_idx]

print("
=== El Punto Dulce Absoluto ===")
print(f"Número óptimo de columnas a retener: {mejor_n}")
print(f"RMSE mínimo alcanzable en este valle: {mejor_rmse:.2f} MPa")

# Graficar zoom al valle de error
plt.figure(figsize=(10, 5))
plt.plot(rango_fino, errores_micro, marker='o', linestyle='-', color='coral', linewidth=2, markersize=6)

# Marcar el punto óptimo
plt.plot(mejor_n, mejor_rmse, marker='*', markersize=18, color='red', label=f'Punto Dulce: {mejor_n} vars ({mejor_rmse:.1f} MPa)')
plt.axvline(x=mejor_n, color='crimson', linestyle='--', alpha=0.5)

plt.title('Micro-Barrido XGBoost: Zoom al Valle del Error (50-71 Variables)', fontsize=14)
plt.xlabel('Cantidad de Variables Retenidas', fontsize=12)
plt.ylabel('Error RMSE (MPa)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Paso 9: Entrenamiento del Modelo de Producción y Serialización

Con el punto dulce establecido en **51 variables**, se procede a:
1. Filtrar las matrices de entrenamiento y prueba a las 51 características óptimas.
2. Entrenar el modelo final `XGBRegressor` con semilla fijada.
3. Evaluar las métricas de validación final ($R^2 pprox 0.7693$, $RMSE pprox 268.45$ MPa).
4. Serializar el estimador entrenado en archivo binario `xgboost_Limite_Elastico_YS_MPa.pkl` mediante `joblib`.
5. Exportar el archivo `variables_Limite_Elastico_YS_MPa.json` con la lista exacta de características requeridas y las métricas oficiales, asegurando su compatibilidad con la API web (`Front/app.py`).

In [ ]:
# 0. Localizar carpeta de exportación en entorno local (Models en la raíz o models local)
posibles_carpetas_modelos = [
    Path("../Models"),
    Path("Models"),
    Path("models")
]
carpeta_modelos = next((p for p in posibles_carpetas_modelos if p.is_dir()), None)
if carpeta_modelos is None:
    carpeta_modelos = Path("models")
    carpeta_modelos.mkdir(parents=True, exist_ok=True)

# 1. Fijar el punto dulce descubierto (51 variables)
n_vars = 51
columnas_top_51 = X.columns[indices_ordenados[:n_vars]]

# 2. Construir el dataset de producción optimizado
X_train_final = X_train[columnas_top_51]
X_test_final = X_test[columnas_top_51]

# 3. Entrenar el modelo Campeón
print(f"Entrenando modelo XGBoost para '{variable_objetivo}' ({n_vars} variables)...")
xgb_final = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_final.fit(X_train_final, y_train)

# 4. Evaluar métricas finales sobre el conjunto de prueba
predicciones_finales = xgb_final.predict(X_test_final)
r2_final = float(r2_score(y_test, predicciones_finales))
rmse_final = float(np.sqrt(mean_squared_error(y_test, predicciones_finales)))

# Imprimir métricas en consola
print("
=== Rendimiento del Modelo de Producción ===")
print(f"Variable Objetivo:       {variable_objetivo}")
print(f"R2 (Varianza Explicada): {r2_final:.4f}")
print(f"RMSE (Error Mínimo):     {rmse_final:.2f} MPa\n")

# 5. Exportar el Modelo serializado
nombre_modelo = carpeta_modelos / f"xgboost_{variable_objetivo}.pkl"
joblib.dump(xgb_final, nombre_modelo)

# 6. Preparar el diccionario con la metadata completa (Métricas + Lista de Variables)
lista_variables = columnas_top_51.tolist()

metadata_exportacion = {
    "variable_objetivo": variable_objetivo,
    "rendimiento": {
        "R2": round(r2_final, 4),
        "RMSE": round(rmse_final, 2)
    },
    "cantidad_variables": n_vars,
    "variables_entrada": lista_variables
}

# 7. Exportar metadata como archivo JSON
nombre_json = carpeta_modelos / f"variables_{variable_objetivo}.json"
with open(nombre_json, "w", encoding="utf-8") as f:
    json.dump(metadata_exportacion, f, indent=4)

# Confirmación de guardado
print(f"¡Éxito! Modelo exportado como: '{nombre_modelo.resolve()}'")
print(f"¡Metadata exportada como: '{nombre_json.resolve()}'\n")

print(f"=== Variables de Entrada Requeridas ({n_vars}) ===")
for i, var in enumerate(lista_variables, 1):
    print(f"{i}. {var}")